In [ ]:
import sys
sys.path.append("./Banksy_py-Bansky_1.2.1")

import os
import anndata as ad
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import glob
from banksy_utils.load_data import load_adata
from banksy_utils.filter_utils import filter_cells
from banksy_utils.filter_utils import normalize_total, filter_hvg
from banksy.main import median_dist_to_nearest_neighbour
from banksy.initialize_banksy import initialize_banksy
from banksy.main import concatenate_all
from banksy.embed_banksy import generate_banksy_matrix
from banksy_utils.umap_pca import pca_umap
from banksy.cluster_methods import run_Leiden_partition
from banksy.plot_banksy import plot_results
from harmony import harmonize
from os.path import join
import umap

def run_banksy_all(adata):
    spatial = adata.obsm["spatial"]
    adata.obs["x"] = spatial[:, 0]
    adata.obs["y"] = spatial[:, 1]
    # Staggering
    coords_df = pd.DataFrame(adata.obs[['x', 'y', 'Slice']])
    coords_df['x'] = coords_df.groupby('Slice')['x'].transform(lambda x: x - x.min())
    global_max_x = max(coords_df['x']) * 1.5
    #Turn samples into factors
    coords_df['sample_no'] = pd.Categorical(coords_df['Slice']).codes
    #Update x coordinates
    coords_df['x'] = coords_df['x'] + coords_df['sample_no'] * global_max_x
    #Update staggered coords to AnnData object
    adata.obs['x_pixel'] = coords_df['x']
    adata.obs['y_pixel'] = coords_df['y']
    adata = normalize_total(adata) 

    adata, adata_allgenes = filter_hvg(adata, n_top_genes=2000, flavor="seurat")
    ## banksy parameters ##
    from banksy.initialize_banksy import initialize_banksy
    coord_keys = ('x_pixel', 'y_pixel', 'coord_xy')
    nbr_weight_decay = 'scaled_gaussian'
    k_geom = 18
    x_coord, y_coord, xy_coord = coord_keys[0], coord_keys[1], coord_keys[2]


    from banksy.main import concatenate_all
    from banksy.embed_banksy import generate_banksy_matrix
    resolutions = [0.40] # clustering resolution for UMAP
    pca_dims = [20] # Dimensionality in which PCA reduces to
    lambda_list = [0.2] # list of lambda parameters
    m = 1
    from banksy_utils.umap_pca import pca_umap
    from banksy.cluster_methods import run_Leiden_partition
    from banksy.plot_banksy import plot_results
    c_map =  'tab20' # specify color map


    ### run banksy ###


    # Include spatial coordinates information
    raw_y, raw_x = adata.obs[y_coord], adata.obs[x_coord]
    adata.obsm[xy_coord] = np.vstack((adata.obs[x_coord].values, adata.obs[y_coord].values)).T

    banksy_dict = initialize_banksy(adata,
                                    coord_keys,
                                    k_geom,
                                    nbr_weight_decay=nbr_weight_decay,
                                    max_m=m,
                                    plt_edge_hist= True,
                                    plt_nbr_weights= True,
                                    plt_agf_angles=False,
                                    plt_theta=False
                                )

    banksy_dict, banksy_matrix = generate_banksy_matrix(adata,
                                                        banksy_dict,
                                                        lambda_list,
                                                        max_m=m)


    banksy_dict["nonspatial"] = {
        # Here we simply append the nonspatial matrix (adata.X) to obtain the nonspatial clustering results
        0.0: {"adata": concatenate_all([adata.X], 0, adata=adata), }
    }
    pca_umap(banksy_dict,
            pca_dims=pca_dims,
            add_umap=False
            )
    from harmony import harmonize
    import umap

    for pca_dim in pca_dims:
        # Run Harmony on PCA
        Z = harmonize(banksy_dict[nbr_weight_decay][0.2]["adata"].obsm[f'reduced_pc_{pca_dim}'], 
                    banksy_dict[nbr_weight_decay][0.2]["adata"].obs, batch_key = 'Slice')
        
        print(f'Replacing adata.obsm["reduced_pc_{pca_dim}"] with harmony corrected embeddings.')
        banksy_dict[nbr_weight_decay][0.2]["adata"].obsm[f'reduced_pc_{pca_dim}'] = Z
        
    return banksy_dict[nbr_weight_decay][0.2]["adata"].obsm[f'reduced_pc_{pca_dim}']


def run_banksy(adata):
    # 检查 spatial 坐标是否存在
    if "spatial" not in adata.obsm or adata.obsm["spatial"] is None:
        print(f"❗ Warning: No spatial info in {file_path}")

    # 添加 x, y 坐标列
    spatial = adata.obsm["spatial"]
    if spatial.shape[1] < 2:
        print(f"❗ Warning: Spatial info does not have 2 coordinates in {file_path}")

    adata.obs["x"] = spatial[:, 0]
    adata.obs["y"] = spatial[:, 1]
    coord_keys = ["x", "y", "spatial"]

    # BANKSY 参数
    k_geom = 15
    max_m = 1
    nbr_weight_decay = "scaled_gaussian"

    resolutions = [0.7]
    pca_dims = [20]
    lambda_list = [0.2]

    # Normalize & HVG filtering
    adata = normalize_total(adata)
    adata, adata_allgenes = filter_hvg(adata, n_top_genes=2000, flavor="seurat")


    nbrs = median_dist_to_nearest_neighbour(adata, key=coord_keys[2])
    banksy_dict = initialize_banksy(adata,
                                    coord_keys,
                                    k_geom,
                                    nbr_weight_decay=nbr_weight_decay,
                                    max_m=max_m,
                                    plt_edge_hist=False,
                                    plt_nbr_weights=False,
                                    plt_agf_angles=False)



    banksy_dict, banksy_matrix = generate_banksy_matrix(adata,
                                                            banksy_dict,
                                                            lambda_list,
                                                            max_m)

    pca_umap(banksy_dict, pca_dims=pca_dims, add_umap=True)
    return banksy_dict[nbr_weight_decay][0.2]["adata"].obsm[f'reduced_pc_{pca_dim}']

In [ ]:
save_path = "../../../results/embeddings/BANKSY"
data_dir = "../../../data/processed/"

In [ ]:
data_names = [
    "Misar-Stereo", "DBiT-RnaAtac-Atac", "Mux-VisiumHD-HE", "Embryo",
    "Misar-E13-E15-E18", "Misar-E15-E18-E18",
    "Lymph-S1-S2-S3", "Lymph-S1-S2-S2",
    "Tonsil-S1-S2-S3", "Tonsil-S1-S2-S2",
    "MB-5M", "MB-H3K4me3"
]

for data_name in data_names:
    data_path = join(data_dir, data_name)
    files = glob.glob(data_path+"/*/*_rna.h5ad")
    adatas = []
    for file in files:
        adata = sc.read_h5ad(file)
        if "Slice" not in adata.obs.columns:
            fallback_slice = file.split("/")[-2]
            adata.obs["Slice"] = fallback_slice

        adatas.append(adata)
    adata = sc.concat(adatas)
    adata.obs["Slice"] = adata.obs["Slice"].astype("category")
    if len(adatas) >1:
        emb = run_banksy_all(adata)
    else:
        emb = run_banksy(adata)
        
    os.makedirs(join(save_path, data_name), exist_ok=True)
    emb = pd.DataFrame(emb, index=adata.obs_names, columns=[i for i in range(emb.shape[1])])
    emb.to_csv(join(save_path, data_name, 'df_emb.csv'))

In [ ]:
base_path = join(data_dir, 'Simulations')
data_names = ["rna-0.1-12-adt-2.1", "rna-0.2-10-adt-2.2", "rna-0.3-7-adt-2.3", "rna-0.4-5-adt-2.4", "rna-0.5-4-adt-2.5"]

for data_name in data_names:
    for seed in ["1", "2", "3"]:
        data_path = join(base_path, data_name, seed)
        files = glob.glob(data_path+"/*_rna*.h5ad")
        # print(files)
        adatas = []
        for file in files:
            adata = sc.read_h5ad(file)
            adata.X = adata.layers["counts"]
            if "Slice" not in adata.obs.columns:
                fallback_slice = file.split("/")[-1].split(".")[0]
                # print(fallback_slice)
                adata.obs["Slice"] = fallback_slice

            adatas.append(adata)
        adata = sc.concat(adatas)
        adata.obs["Slice"] = adata.obs["Slice"].astype("category")
        if len(adatas) >1:
            emb = run_banksy_all(adata)
        else:
            emb = run_banksy(adata)
        os.makedirs(join(save_path, 'Simulations', data_name, seed), exist_ok=True)
        emb = pd.DataFrame(emb, index=adata.obs_names, columns=[i for i in range(emb.shape[1])])
        emb.to_csv(join(save_path, 'Simulations', data_name, seed, 'df_emb.csv'))